# 04 — Driving the system with `%agent` magics

This notebook runs on the **`agent-kernel`** kernel itself (not `python3`). Install it once with:

```bash
python -m agent_kernel install --user
```

Then in JupyterLab choose the `agent-kernel` kernel for this notebook.

Under the agent-kernel:
- Cells starting with `%agent ...` are dispatched through `agent_kernel.magics`.
- Any other cell executes as normal Python (the kernel subclasses `IPythonKernel`).
- A single `AgentKernel` instance is bound per kernel process, scoped to the
  workspace from `$AGENT_KERNEL_WORKSPACE` (defaults to the kernel's cwd).

> If you opened this notebook on `python3` by mistake, every `%agent` cell will
> fail with `UsageError: Line magic function '%agent' not found.` That's the
> signal to switch kernels via *Kernel → Change Kernel*.

## 1. What magics are available?

`%agent help` lists every dispatcher entry. The handlers are line-oriented, shlex-parsed, and return JSON for easy programmatic consumption.

In [ ]:
%agent help

## 2. Inspect policy and quota

In [ ]:
%agent policy show

In [ ]:
%agent quota

## 3. Author a notebook on disk, then create + run a task for it

Because we are on the `agent-kernel` (an IPython subclass), regular Python still works for setup.

In [ ]:
from pathlib import Path
import nbformat
from nbformat.v4 import new_notebook, new_code_cell

nb = Path.cwd() / 'demo.ipynb'
nbformat.write(new_notebook(
    cells=[new_code_cell("print('hello from the child notebook')")],
    metadata={'kernelspec': {'name': 'python3', 'display_name': 'Python 3'}},
), nb)
print('wrote', nb)

In [ ]:
%agent task new demo.ipynb --kernel python3

Grab the `task_id` from the JSON output above and run it. (In a real workflow you'd capture it via Python; for the demo, copy-paste.)

In [ ]:
# Show all task ids currently known to the workspace
%agent task list

Use one of those ids in the next two cells. Magics accept a single positional argument:

In [ ]:
# Replace TASK_ID with one from the list above before executing.
# %agent task status TASK_ID
# %agent run TASK_ID

## 4. Tail the ledger

`%agent ledger tail N` returns the last N events from the JSONL store. Add `--task TASK_ID` to scope to a single task.

In [ ]:
%agent ledger tail 10

## 5. Spawn a child from a template

`%agent spawn <parent_task_id> <template> [--param k=v]... [--kernel python3]` uses the spawn manager from notebook 03 under the hood. Parameters are parsed as JSON when possible, otherwise as strings.

```
%agent spawn TASK_ID python-analysis --param query="recent papers" --param limit=5
```

## What the kernel is **not** doing yet

Right now the kernel is a magic dispatcher plus normal Python. It does **not** yet drive an LLM-backed ReAct loop on your behalf — you compose the orchestration explicitly via magics or via the Python API.

The substrate for a ReAct kernel is all here (task lifecycle, spawn lineage, structured LLM with budget accounting, append-only provenance); adding a `ReActPolicy` runtime + `%agent agent run` magic is the next milestone. See the *What's next* section of `docs/getting-started.md`.